In [61]:
#load necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob as glob
from pathlib import Path


In [ ]:
# collect all raw CSV file paths from the neso directory
files = glob.glob('/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/*.csv')

files

['/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2022.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2023.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2021.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2009.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2020.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2018.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecasting/data/raw/neso/demanddata_2024.csv',
 '/Users/kwakye/Desktop/Github Projects/portfolio_project_one/gb-demand-imbalance-forecast

In [ ]:
# extract year from each filename e.g. demanddata_2025.csv -> '2025'
years = [Path(f).stem.split('_')[-1] for f in files]
years

['2022',
 '2023',
 '2021',
 '2009',
 '2020',
 '2018',
 '2024',
 '2025',
 '2019',
 '2017',
 '2016',
 '2014',
 '2015',
 '2011',
 '2010',
 '2012',
 '2013']

In [ ]:
# count columns per year to detect schema changes across the dataset
column_count = {}
for i,f in enumerate(files):
    df = pd.read_csv(f)
    column_count[years[i]] = len(df.columns.tolist())

# print sorted by year to see where schema changes occur
for year, count in sorted(column_count.items()):
    print(year, count)

2009 17
2010 17
2011 17
2012 17
2013 17
2014 17
2015 17
2016 17
2017 17
2018 17
2019 21
2020 21
2021 21
2022 21
2023 22
2024 22
2025 22


In [65]:
#2009 to 2018 share the same number of columns,
#2019 to 2021 share the same number of columns,,
#2022 to 2025 share the same number of columns 

group_2009_2018 = [file for file, year in zip(files, years) 
                   if 2009 <= int(year) <= 2018]
group_2019_2022 = [file for file, year in zip(files, years) 
                   if 2019 <= int(year) <= 2022]
group_2023_2025 = [file for file, year in zip(files, years) 
                   if 2023 <= int(year)]

In [ ]:
# assert all files in a group share identical column names
# uses nrows=0 to read only the header row — avoids loading full data
def check_columns(file_path):
    reference_cols = pd.read_csv(file_path[0], nrows=0).columns.tolist()
    for f in file_path:
        df_cols = pd.read_csv(f, nrows=0).columns.tolist()
        assert reference_cols == df_cols, f"Column mismatch in {Path(f).name}"
    print(f"All {len(file_path)} files have identical columns.")

In [67]:
#check columns
check_columns(group_2009_2018)
check_columns(group_2019_2021)
check_columns(group_2022_2025)

All 10 files have identical columns.
All 4 files have identical columns.
All 3 files have identical columns.


In [68]:
#load 2009, 2019 and 2025 demand data
df_2009 = pd.read_csv(files[3],nrows = 0).columns.tolist()
df_2019 = pd.read_csv(files[8], nrows = 0).columns.tolist()
df_2025 = pd.read_csv(files[7], nrows = 0 ).columns.tolist()

In [ ]:
# intersect columns across all three schema versions to find shared columns
common_cols = set(df_2009) & set(df_2019) & set(df_2025)
print(len(common_cols))
common_cols

17


{'BRITNED_FLOW',
 'EAST_WEST_FLOW',
 'EMBEDDED_SOLAR_CAPACITY',
 'EMBEDDED_SOLAR_GENERATION',
 'EMBEDDED_WIND_CAPACITY',
 'EMBEDDED_WIND_GENERATION',
 'ENGLAND_WALES_DEMAND',
 'IFA2_FLOW',
 'IFA_FLOW',
 'MOYLE_FLOW',
 'ND',
 'NEMO_FLOW',
 'NON_BM_STOR',
 'PUMP_STORAGE_PUMPING',
 'SETTLEMENT_DATE',
 'SETTLEMENT_PERIOD',
 'TSD'}

In [ ]:
# union all columns then subtract common ones to find columns added/removed across schemas
all_cols = set(df_2009) | set(df_2019) | set(df_2025)

different_cols = all_cols - common_cols

different_cols

{'ELECLINK_FLOW',
 'GREENLINK_FLOW',
 'NSL_FLOW',
 'SCOTTISH_TRANSFER',
 'VIKING_FLOW'}

In [ ]:
# preserve original column order from the 2009 schema
common_cols = [col for col in df_2009 if col in common_cols]
common_cols

['SETTLEMENT_DATE',
 'SETTLEMENT_PERIOD',
 'ND',
 'TSD',
 'ENGLAND_WALES_DEMAND',
 'EMBEDDED_WIND_GENERATION',
 'EMBEDDED_WIND_CAPACITY',
 'EMBEDDED_SOLAR_GENERATION',
 'EMBEDDED_SOLAR_CAPACITY',
 'NON_BM_STOR',
 'PUMP_STORAGE_PUMPING',
 'IFA_FLOW',
 'IFA2_FLOW',
 'BRITNED_FLOW',
 'MOYLE_FLOW',
 'EAST_WEST_FLOW',
 'NEMO_FLOW']

In [ ]:
# load all years keeping only common columns, then concatenate into one DataFrame
dataframes = []
for f in files:
    df = pd.read_csv(f, usecols=common_cols)
    dataframes.append(df)

combined_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
# check total rows and columns in combined dataset
combined_df.shape

(298032, 17)

In [ ]:
# check for missing values across all columns
combined_df.isnull().sum()

SETTLEMENT_DATE              0
SETTLEMENT_PERIOD            0
ND                           0
TSD                          0
ENGLAND_WALES_DEMAND         0
EMBEDDED_WIND_GENERATION     0
EMBEDDED_WIND_CAPACITY       0
EMBEDDED_SOLAR_GENERATION    0
EMBEDDED_SOLAR_CAPACITY      0
NON_BM_STOR                  0
PUMP_STORAGE_PUMPING         0
IFA_FLOW                     0
IFA2_FLOW                    0
BRITNED_FLOW                 0
MOYLE_FLOW                   0
EAST_WEST_FLOW               0
NEMO_FLOW                    0
dtype: int64

In [ ]:
# save combined dataset to processed data directory
combined_df.to_csv("../data/processed/gb_demand_2009_2025.csv", index=False)